## Cross-Country Job Volume

Tech workforce size and share of total employed across USA, India, and China.

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import plotly.express as px

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}

usa = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
india = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_india_data.csv')
china = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_china_data.csv')
df = pd.concat([usa, india, china], ignore_index=True)
vol = df[df['career_stage'] == 'mid'][['country', 'role', 'employed_thousands']].copy()
vol['role_label'] = vol['role'].map(ROLE_LABELS)
country_colors = {'USA': '#2171b5', 'India': '#31a354', 'China': '#e6550d'}

In [2]:
swe_vol = vol[vol['role'] == 'software_engineer'].copy()
fig = px.bar(
    swe_vol, x='country', y='employed_thousands', color='country',
    color_discrete_map=country_colors,
    title='Software Engineer Headcount by Country (2023, thousands)',
    labels={'employed_thousands': 'Employed (thousands)', 'country': 'Country'},
    text='employed_thousands',
)
fig.update_traces(texttemplate='%{text:,}K', textposition='outside')
fig.show()

In [3]:
usa_order = (
    vol[vol['country'] == 'USA']
    .sort_values('employed_thousands', ascending=False)['role_label'].tolist()
)
fig2 = px.bar(
    vol, x='role_label', y='employed_thousands', color='country',
    barmode='group',
    category_orders={'role_label': usa_order, 'country': ['USA', 'India', 'China']},
    color_discrete_map=country_colors,
    title='Total Employed by Role and Country (2023, thousands)',
    labels={'role_label': 'Role', 'employed_thousands': 'Employed (thousands)', 'country': 'Country'},
)
fig2.update_layout(xaxis_tickangle=-35)
fig2.show()

In [4]:
total = vol.groupby('country')['employed_thousands'].sum().reset_index().rename(
    columns={'employed_thousands': 'total_k'}
)
swe = vol[vol['role'] == 'software_engineer'][['country', 'employed_thousands']].rename(
    columns={'employed_thousands': 'swe_k'}
)
share = swe.merge(total, on='country')
share['swe_pct'] = (share['swe_k'] / share['total_k'] * 100).round(2)

fig3 = px.bar(
    share, x='country', y='swe_pct', color='country',
    color_discrete_map=country_colors,
    title='Software Engineers as % of All Tracked Workers by Country (2023)',
    labels={'swe_pct': 'SWE share (%)', 'country': 'Country'},
    text='swe_pct',
)
fig3.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig3.show()